# ఖర్చుల క్లెయిమ్ విశ్లేషణ

ఈ నోట్బుక్ లోకల్ రసీదు చిత్రాల నుండి ప్రయాణ ఖర్చులను ప్రాసెస్ చేయడానికి ప్లగిన్లను ఉపయోగించే ఏజెంట్లను ఎలా సృష్టించాలో, ఖర్చుల క్లెయిమ్ ఇమెయిల్ తయారు చేయడం మరియు ఖర్చుల డేటాను పై చార్ట్ ద్వారా దృశ్యమానపరచడం చూపిస్తుంది. ఏజెంట్లు పని సందర్భం ఆధారంగా ఫంక్షన్లను డైనమిక్‌గా ఎంచుకుంటాయి.

పదజాలాలు:
1. OCR ఏజెంట్ స్థానిక రసీదు చిత్రాన్ని ప్రాసెస్ చేసి ప్రయాణ ఖర్చుల డేటాను పొందుతుంది.
2. ఇమెయిల్ ఏజెంట్ ఖర్చుల క్లెయిమ్ ఇమెయిల్ ను రూపొందిస్తుంది.

### ప్రయాణ ఖర్చుల సందర్భానికి ఉదాహరణ:
మీరు మరో నగరంలో వ్యాపార సమావేశం కోసం ప్రయాణిస్తున్న ఉద్యోగి అని ఊహించండి. మీరు పని చేసే సంస్థ అన్ని అనువైన ప్రయాణ సంబంధ ఖర్చులను తిరిగి చెల్లించే విధంగా పాలసీ కలిగి ఉంది. క్రింది విధంగా సంభవించే ప్రయాణ ఖర్చుల విభజన ఉంది:
- రవాణా:
మీ గృహ నగరంలో నుండి గమ్య నగరానికి రౌండ్ ట్రిప్ సుమారుగా విమాన ప్రయాణం.
విమానాశ్రయం చేరడం/విమానాశ్రయం నుండి వెళ్తున్న టాక్సీ లేదా రైడ్-హైలింగ్ సేవలు.
గమ్య నగరంలోని స్థానిక రవాణా (జనగత రవాణా, కార్లను అద్దెలకు తీసుకోవడం, లేదా టాక్సీలు).

- నివాసం:
సమావేశం జరుగుతున్న ప్రదేశానికి సన్నిహితంగా ఉన్న మిడ్-రేంజ్ వ్యాపార హోటల్లో మూడు రాత్రులు ఉండడం.

- భోజనం:
ఉద్యోగ సంస్థ పిర్ డియం పాలసీ ఆధారంగా ప్రతి రోజు అలవెన్స్ - అల్పాహారం, మధ్యాహ్న భోజనం, రాత్రి భోజనం.

- ఇతర ఖర్చులు:
విమానాశ్రయం వద్ద పార్కింగ్ ఫీజు.
హోటలులో ఇంటర్నెట్ యాక్సెస్ ఛార్జులు.
టిప్పులు లేదా చిన్న సేవా ఛార్జులు.

- డాక్యుమెంటేషన్:
మీరు అన్ని రసీదులు (విమాన టికెట్లు, టాక్సీలు, హోటల్, భోజనాలు, మొదలైనవి) మరియు తిరిగి చెల్లింపుకు పూర్తి చేసిన ఖర్చుల నివేదికను సమర్పిస్తారు.


## అవసరమైన లైబ్రరీలను ఇంపోర్ట్ చేయండి

ఈ నోట్‌బుక్ కోసం అవసరమైన లైబ్రరీలు మరియు మాడ్యూల్స్‌ను ఇంపోర్ట్ చేయండి.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Define Expense Models

 వ్యక్తిగత ఖర్చుల కోసం ఒక Pydantic మోడల్ సృష్టించండి మరియు వినియోగదారు ప్రశ్నను నిర్మితమైన ఖర్చు డేటాకు మార్చడానికి ఒక ExpenseFormatter తరగతిని రూపొందించండి.

 ప్రతి ఖర్చు క్రింది ఫార్మాట్‌లో చూపబడుతుంది:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## టూల్స్ నిర్వచనం - ఇమెయిల్ సృష్టించడం

ఖర్చు క్లెయిమ్ సమర్పించడానికి ఇమెయిల్ సృష్టించే టూల్ ఫంక్షన్‌ని తయారు చేయండి.
- ఈ టూల్ Microsoft Agent Framework నుండి `@tool` డెక్కరేటర్ ఉపయోగిస్తుంది.
- ఖర్చుల మొత్తం మొత్తం లెక్కించి వివరాలను ఇమెయిల్ బాడీగా ఫార్మాట్ చేస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# రశీదు చిత్రాల నుండి ప్రయాణ ఖర్చులను తీసుకునే సాధనం

రశీదు చిత్రాలలో నుంచి ప్రయాణ ఖర్చులను తీసుకునే సాధన ఫంక్షన్ సృష్టించండి.
- ఈ సాధనం Microsoft Agent Framework నుండి `@tool` డెకరేటర్ ఉపయోగిస్తుంది.
- ఇది రశీదు చిత్రాన్ని చదవుతూ, దాన్ని base64గా కుడి చేసి, ఏజెంట్ విశ్లేషించడానికి డేటా URIను తిరిగి ఇస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## ఖర్చుల ప్రాసెసింగ్

ఏజెంట్లను నిర్వచించండి మరియు వాటిని `WorkflowBuilder` ఉపయోగించి ఒక క్రమబద్ధమైన వర్క్‌ఫ్లోలో వైర్ చేయండి.
- OCR ఏజెంట్ రిసీట్లు చిత్రం నుండి నిర్మితమైన ఖర్చుల డేటాను `load_receipt_image` టూల్ ఉపయోగించి తీసుకుంటుంది.
- ఇమెయిల్ ఏజెంట్ తీసుకున్న డేటాను తీసుకుని `generate_expense_email` టూల్ ఉపయోగించి ఒక ప్రొఫెషనల్ ఖర్చుల క్లైమ్ ఇమెయిల్‌ను రూపొందిస్తుంది.
- `WorkflowBuilder` లో `add_edge` తయారు చేయడం క్రమబద్ధమైన పైప్‌లైన్‌ను సృష్టిస్తుంది: OCR ఏజెంట్ → ఇమెయిల్ ఏజెంట్.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

క్రమబద్ధమైన వర్క్‌ఫ్లోని నిర్మించి దాన్ని రీసీట్ చిత్రం ప్రాసెస్ చేయడానికి మరియు ఖర్చు క్లెయిమ్ ఇమెయిల్‌ను సృష్టించడానికిగాను అమలు చేయండి.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**నివేదన**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. కచ్చితత్వానికి మేము ప్రయత్నించినప్పటికీ, స్వయంచాలక అనువాదాల్లో తప్పులు లేదా తప్పుదోషాలు ఉండవచ్చు అని దయచేసి గమనించండి. మూల పత్రం ఆ దాని మాతృభాషలో అధికారం కలిగిన మూలంగా పరిగణించాలి. అత్యవసర సమాచారానికి, నిపుణుల మానవ అనువాదం చేయించుకోవాలని సిఫార్సు చేయబడుతుంది. ఈ అనువాదం ఉపయోగంవల్ల ఉత్పన్నమయ్యే ఏవైనా అపార్థాలు లేదా లోపాలను గురించి మేము బాధ్యత తీసుకోము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
